# Gemma3-4B -- evaluate the ViNumQA program-only SFT adapter

Loads the LoRA adapter produced by `gemma3-4b-stf-wo-reasoning-train-only.ipynb` and scores Program
Accuracy / Execution Accuracy on `test.json`, using **the scorer cell copied verbatim from
`qwen3-4b-stf-wo-reasoning-trace.ipynb`** -- same parser, same executor, so any difference in the
numbers is a difference between the models, not between two scorers.

**Before running:**

1. Add the training notebook's output as a data source (*Add Input > Your Work > Notebooks*). The
   adapter directory is then found by glob; the trainer's `output_dir` (same name without
   `-adapter`, holding only `checkpoint-*`) is skipped because it has no `adapter_config.json`.
2. Attach the ViNumQA dataset for `test.json`.
3. A T4/P100 is enough -- there is no optimiser state here, which is the whole reason this is a
   separate notebook from training.

Generation is one sample at a time. That is not laziness: batching requires left padding, and
Unsloth's fast inference path derives token positions from the cache length rather than from the
attention mask, so padded rows decode at the wrong positions -- measured elsewhere in this repo, that
cost 0.6419 -> 0.6338 PA on a comparable model. The Qwen baseline also generates one at a time, so
this also keeps the two protocols identical.

Generations are checkpointed to `/kaggle/working` every 20 samples and resume automatically, so an
interrupted session continues instead of restarting.

## Held parallel to the with-reasoning arm

The evaluation protocol below is deliberately the same one
`sft-w-reasoning-trace-distill/gemma3-4b-eval-only.ipynb` runs, so the two SFT arms differ only in
what the adapter was trained to emit -- not in how they are measured:

* **Headline number is greedy (`do_sample=False`, `N_VOTES = 1`).** Not left to the model's own
  `generation_config`: `gemma-3-4b-it` ships one carrying `top_k=64 / top_p=0.95`, so an implicit
  `do_sample` would make the headline PA/EA a sampled draw that does not reproduce between runs.
  Sampling (Gemma-3's recommended `1.0 / 0.95 / 64`) is used only when voting needs it.
* **`N_VOTES > 1` turns the same loop into self-consistency,** voting on the *normalised* program
  (`program_tokenization`) so cosmetically different but structurally identical programs share a
  vote. Extra votes come from `num_return_sequences`, never from batching prompts -- one prompt means
  nothing to pad, and memory stays bounded by `N_VOTES` rather than by `N_VOTES x batch`.
* **Checkpoints and result files are keyed by `ADAPTER_VARIANT` and `k`,** so a k=1 run and a later
  k=5 run do not overwrite each other's partial generations.
* **A per-sample `OutOfMemoryError` is caught and scored 0** rather than killing a run that is
  already hours in.
* **The self-consistency retest at the end** re-runs only the samples that scored 0 on both metrics,
  which answers whether the remaining errors are sampling noise or systematic.

**The one deliberate departure:** no `</think>` handling anywhere. The with-reasoning sibling decodes
to text and cuts at the last `</think>`; this adapter was trained on program-only labels, so
everything past the prompt *is* the program. Searching for a tag that never appears could only
discard a correct generation.

## Gemma-specific mechanics

None of these are stylistic. Each was an outright failure -- a crash, a hang or a silent no-op --
when the Qwen setting was carried over to `gemma-3-4b-it`:

1. **`UNSLOTH_ENABLE_FLEX_ATTENTION=0` + `attn_implementation="eager"`,** set *before* `import
   unsloth`. gemma3 is in Unsloth's `_FLEX_PREFERRED_MODELS`; that kernel does not compile on a T4
   (sm75) and the `sdpa_dense` fallback materialises the full `(B, H, L, L)` score matrix. SDPA is
   not reachable either (gemma3 is in `DISABLE_SDPA_MODEL_NAMES`), so eager is not a downgrade.
2. **`FastModel` + `finetune_*` flags, not `FastLanguageModel` + `target_modules`.**
   `unsloth/gemma-3-4b-it` is a `Gemma3ForConditionalGeneration` (text decoder + SigLIP vision
   tower); the q/k/v/o and MLP names in the Qwen list also exist inside that tower, so passing the
   list would attach LoRA to vision modules a text-only task never trains.
3. **`SFTTrainer` is given `text_tokenizer`, not the processor.** Unsloth returns a
   `Gemma3Processor`; handing it to TRL builds a collator exposing `.image_processor`, and
   `unsloth_zoo`'s `_is_vision_collator()` then makes `train_on_responses_only` refuse the trainer.
4. **Mask markers are `<start_of_turn>user\n` / `<start_of_turn>model\n`,** not ChatML's
   `<|im_start|>...`. Those tokens do not exist in Gemma's vocabulary, so every label would be
   masked out.
5. **`.removeprefix("<bos>")` after templating.** Gemma's template emits `<bos>` itself and
   `SFTTrainer` re-tokenizes with `add_special_tokens=True`; without this every training sequence
   starts with a double BOS. The eval side of the same rule is `add_special_tokens=False` when
   encoding, so the prompt carries exactly one `<bos>` -- the count training saw.
6. **`eos_token_id` must include `<end_of_turn>`.** Gemma-3 ends a turn with `<end_of_turn>`, not
   `<eos>`, and `generate()` only stops on ids listed in `eos_token_id`. Without it every sample
   burns the full token budget -- correct output, several times the runtime.
7. **No `enable_thinking` and no system slot.** Gemma's template has no thinking mode (so there is
   no `</think>` to strip, unlike the Qwen notebook) and no `system` role; it folds the system turn
   into the head of the first user turn, so the prompt text stays identical to the baseline.
8. **`tokenizer` is a `Gemma3Processor`.** Its `__call__` takes `images` first, so
   `tokenizer([text])` binds the prompt to the wrong argument, and `apply_chat_template(...,
   tokenize=True)` raises `TypeError: string indices must be integers` on string content.
   Templating stays on the processor at `tokenize=False`; tokenizing and decoding go through
   `text_tokenizer`, the unwrapped inner tokenizer.
9. **Gemma-3 sampling defaults:** `temperature=1.0, top_p=0.95, top_k=64`. Qwen's `0.7 / 0.8 / 20`
   are Qwen's own recommended values; each model uses what its authors published, as elsewhere in
   this repo. These apply to the `N_VOTES > 1` and retest paths only -- the headline run is greedy.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Environment

Must be set before `import unsloth` in the next cell. There is no backward pass here, so
`flex_attention` would not OOM the way training did -- it would just turn 497 sequential generations
into an uncompiled, op-by-op crawl.

In [ ]:
import os

# THE OOM/SPEED FIX, and it only works before `import unsloth`. Unsloth lists gemma3 in
# _FLEX_PREFERRED_MODELS, so it selects flex_attention by default. That kernel does not compile on a
# T4 (sm75), so torch falls back to `sdpa_dense` -- the eager Python decomposition, which
# materialises the full (B, H, L, L) score matrix AND its gradient. On the with-reasoning sibling
# that produced `OutOfMemoryError: Tried to allocate 1.17 GiB` inside sdpa_dense_backward, plus
# ~0.01 it/s because the fallback runs uncompiled, op by op.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"

# Kaggle's "GPU T4 x2" exposes 2 devices. Free-tier Unsloth does not support multi-GPU training and
# HF Trainer would otherwise wrap the model in DataParallel.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# The allocator hint the OOM message itself suggests: long, highly variable sequence lengths
# fragment the pool badly, and expandable segments let freed blocks be reused at a different size.
# (PyTorch renamed this variable, so set both and let the version in use read whichever it knows.)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

### Load the fine-tuned adapter

In [ ]:
from unsloth import FastModel
import torch
import glob, os

MAX_SEQ_LENGTH = 2048   # matches the training notebook and the Qwen baseline

# The label every artifact this notebook writes is keyed by, mirroring ADAPTER_VARIANT in
# `sft-w-reasoning-trace-distill/gemma3-4b-eval-only.ipynb`. It is a fixed string here rather than a
# knob: that arm trains several dataset variants ("pa_en", "v6", ...) and tags each adapter
# directory with the one it came from, whereas the program-only arm has exactly one adapter
# (`gemma3-4b-vinumqa-sft-wo-reasoning-adapter`). Keeping the name means the two notebooks produce
# artifacts that sort and compare side by side.
ADAPTER_VARIANT = "wo_reasoning"


def _find_adapter():
    """First directory that both matches and actually holds adapter weights.

    Checking for adapter_config.json matters: the training notebook writes TWO directories, and the
    trainer's output_dir (`gemma3-4b-vinumqa-sft-wo-reasoning`, no `-adapter`) holds only checkpoint-*
    subfolders and a model card.
    """
    patterns = [
        "/kaggle/input/**/gemma3-4b-vinumqa-sft-wo-reasoning-adapter",
        "/kaggle/input/**/*sft-wo-reasoning-adapter*",
        "/kaggle/input/**/*vinumqa*adapter*",
    ]
    for pattern in patterns:
        for path in sorted(glob.glob(pattern, recursive=True)):
            if os.path.isfile(os.path.join(path, "adapter_config.json")):
                return path
    return None


ADAPTER_DIR = _find_adapter()
assert ADAPTER_DIR, (
    "No directory containing adapter_config.json found under /kaggle/input. Attach the training "
    "notebook's output as a data source (Add Input > Your Work > Notebooks), or set ADAPTER_DIR "
    "manually to the folder shown in the Data panel."
)
print("Found adapter:", ADAPTER_DIR)
print("  contents:", sorted(os.listdir(ADAPTER_DIR)))

# The catch-all patterns above would also match the with-reasoning arm's adapter if both are
# attached at once -- and that one emits a <think> block this notebook does not strip, so the mix-up
# would show up as a mysteriously near-zero PA rather than as an error.
assert "wo-reasoning" in ADAPTER_DIR or "wo_reasoning" in ADAPTER_DIR, (
    f"{ADAPTER_DIR} does not look like the program-only adapter. If both SFT arms are attached, "
    f"set ADAPTER_DIR manually to the '-sft-wo-reasoning-adapter' directory."
)

# FastModel, NOT FastLanguageModel -- the same correction the training notebook needed.
# from_pretrained on an adapter directory reads base_model_name_or_path out of adapter_config.json,
# pulls the base model, and applies the LoRA weights on top.
model, tokenizer = FastModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    attn_implementation = "eager",   # see the env-var cell above
)
FastModel.for_inference(model)

# `tokenizer` is a Gemma3Processor again (the training notebook saved the whole processor). Its
# __call__ takes `images` first, so every direct tokenize/decode below goes through text_tokenizer;
# apply_chat_template stays on the processor, which carries the pinned gemma-3 template.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

_EOS_IDS = []
for _t in (text_tokenizer.eos_token_id, text_tokenizer.convert_tokens_to_ids("<end_of_turn>")):
    if _t is not None and _t not in _EOS_IDS:
        _EOS_IDS.append(_t)

_impl = getattr(model.config, "_attn_implementation", None) or getattr(model.config, "attn_implementation", None)
print(f"Adapter loaded. model={type(model).__name__}, attn={_impl}, eos_ids={_EOS_IDS}")
print(f"tokenizer: {type(tokenizer).__name__}  ->  text_tokenizer: {type(text_tokenizer).__name__}")
assert _impl != "flex_attention", (
    "flex_attention is selected -- generation will crawl on a T4. Restart the kernel so "
    "UNSLOTH_ENABLE_FLEX_ATTENTION=0 is read before unsloth is imported."
)
assert getattr(tokenizer, "chat_template", None), (
    "No chat template on the loaded processor -- prompts would not carry the <start_of_turn> markers "
    "the adapter was trained on. Check that chat_template.jinja is in the adapter directory."
)

### Data + prompt (identical to the training notebook)

Both cells below are copied verbatim from `qwen3-4b-stf-wo-reasoning-trace.ipynb`, which is also
where the training notebook takes them from -- the prompt the model is evaluated on is byte-identical
to the one it was trained on, and to the one the Qwen baseline used.

In [ ]:
import glob
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# Same dataset as the Qwen baseline. The explicit path is the one that notebook hardcodes; the glob
# fallback finds the files wherever the dataset happens to be mounted.
_CANDIDATES = [
    Path("/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa"),
    Path("/kaggle/input/vlsp2025-vinumqa"),
    Path("datasets/ViNumQA"),   # local repo path, if running outside Kaggle
]
DATA_DIR = next((p for p in _CANDIDATES if (p / "train.json").exists()), None)
if DATA_DIR is None:
    _hits = glob.glob("/kaggle/input/**/train.json", recursive=True)
    DATA_DIR = Path(_hits[0]).parent if _hits else None
if DATA_DIR is None:
    raise FileNotFoundError(
        "train.json not found -- attach the ViNumQA dataset (ntphuc149x2/vlsp2025-vinumqa) "
        "as a data source, or add its path to _CANDIDATES above."
    )
print("Using data from:", DATA_DIR)

train_df = pd.read_json(DATA_DIR / "train.json")
valid_df = pd.read_json(DATA_DIR / "valid.json")
test_df = pd.read_json(DATA_DIR / "test.json")
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")

# The full split, not a trace-filtered subset. The train_with_reasoning_trace_* files are built by
# dropping samples a teacher could not solve; using one here would make the comparison measure the
# data filter instead of the training signal.
assert (len(train_df), len(valid_df), len(test_df)) == (2993, 584, 497), (
    f"Unexpected split sizes {(len(train_df), len(valid_df), len(test_df))} -- expected "
    "(2993, 584, 497). Point DATA_DIR at the full split, not a *_with_reasoning_trace_* subset."
)

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["table_raw"] = df["table"]  # keep the raw rows for table_* row-name lookup at eval time
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
    return df

train_df = process_split(train_df)
valid_df = process_split(valid_df)
test_df = process_split(test_df)
test_df["generated_program"] = ""

train_df.sample(n=3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so results are
# comparable across all experiments (only the training regime differs).

<a name="Eval"></a>
### PA / EA

In [ ]:
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented:

  * Program Accuracy is *symbolic* equivalence, via sympy, between the gold and
    predicted expressions -- not a string or structural match. A prediction may
    reorder or restructure the arithmetic, but it may only use literals that
    appear in the gold program, so it cannot invent constants such as the `100`
    of a percentage rescaling.
  * Execution Accuracy compares the executed result to `exe_ans` exactly, after
    rounding to 5 decimals. No tolerance.
  * `greater` yields the strings "yes"/"no", matching how the dataset stores
    those answers.
  * Every step takes exactly two arguments. Verified against the data: all 663
    steps across the gold programs are binary, and `table_*` always takes a row
    label plus `none` (454 occurrences) rather than a list of values (2).
  * FinQA's `const_` tokens are still understood.

Five corrections are applied, each because the unmodified script cannot
reproduce ViNumQA's own gold, not because the protocol was thought wrong:

1. Tokenisation of bracketed row labels. `program_tokenization` splits on every
   bracket, so `table_min(ROE (%), none)` shatters into six tokens and fails the
   four-tokens-per-step structure check. 35 of the 497 test programs name a row
   whose label contains brackets -- `ROE (%)`, `EPS (VND)`, `P/E (x)` -- and all
   35 were unscoreable. Tokenisation is now bracket-depth aware.

2. Accounting negatives. Tables write negative amounts as `(3344)`. The original
   `process_row` takes the text before the first bracket, leaving an empty
   string, so the cell fails to parse. The dataset's own `exe_ans` was computed
   with those values -- e.g. `table_min(LN hoạt động (tỷ đồng), none)` expects
   -3344 from a row holding `(3344)`. The `-1046 ( 1046 )` form the original
   handled correctly is unchanged.

3. Unparseable cells no longer void the whole row. Measured over the 393 gold
   `table_*(<row>, none)` programs in train, skipping such cells reproduces
   `exe_ans` for 386 against 381 when the row is voided, so skipping is what the
   dataset was built with.

4. `exe_ans` is stored as a string here ("31.0") where FinQA stores a number, so
   the comparison `exe_res == gold_res` was never true. It is coerced, leaving
   the "yes"/"no" answers alone.

5. The `assert exe_res == gold_res` inside the program-accuracy branch is
   dropped. It is a debug check, and a single rounding disagreement aborts the
   whole evaluation.

`evaluate_result_official` runs the unmodified protocol for comparison, so the
cost of each correction can be seen rather than assumed.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


# ------------------------------------------------------------------ numbers --
def str_to_num(text: str) -> Union[float, str]:
    """FinQA's literal parser, unchanged: returns "n/a" rather than raising."""
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            # Multiples are written "14.3x" in these tables. Unlike "%", the
            # suffix carries no scaling -- the gold answer for such a row is the
            # plain multiple.
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    """Parse one table cell.

    Adds the `(3344)` form to what the original handled; `$ -1046 ( 1046 )`
    still resolves through the original's "text before the first bracket" rule.
    """
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    """Numeric values of a table row, or "n/a" if the row cannot be reduced.

    A cell that merely marks a missing period ("-", "NA", an em dash) is
    skipped: rows in this dataset routinely lack a year or two, and voiding the
    whole row over one gap loses reductions the gold answers depend on. A cell
    with real but unreadable content still voids the row, so genuine parse
    failures are not silently averaged away.
    """
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


# -------------------------------------------------------------- tokenisation --
def program_tokenization(original_program: str) -> List[str]:
    """Tokenise into ['op(', arg1, arg2, ')', ..., 'EOF'].

    Bracket-depth aware, so a row label like `ROE (%)` stays one token. The
    original split on every bracket, which shattered such labels and broke the
    four-tokens-per-step structure the rest of the protocol relies on.

    Raises ValueError if trailing, non-whitespace text remains once no further
    step can be parsed (e.g. a step missing its closing paren, which happens
    both in a handful of gold programs and -- more importantly -- in model
    generations cut off by a max_new_tokens limit). An earlier version of this
    tokenizer silently stopped and returned only the steps parsed so far,
    which let a truncated program like "subtract(100, 50), divide(#0, 5"
    (missing text and closing paren) score as a valid, complete one-step
    program instead of being rejected -- a false positive for exactly the kind
    of generation failure this evaluator needs to catch.
    """
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching ')' found) in program: '{original_program}'"
            )

        program.append(m.group(1) + "(")
        # Split arguments on depth-0 commas so brackets inside a label survive.
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: '{text[pos:]}' (from: '{original_program}')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    """Recover a program string from raw model output.

    Bracket matched, so an outer call is never silently discarded: the earlier
    regex could only match a bracket-free call, so `multiply(divide(a, b), 100)`
    was reduced to its inner `divide(a, b)` and a percentage rescaling scored as
    if it were the gold answer.
    """
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text



def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    """Group a tokenised program into (op, arg1, arg2) triples.

    The original walked the token list by joining it and splitting on ")", which
    silently mis-splits any argument containing a bracket -- exactly the row
    labels this dataset uses, e.g. `EPS (VND)`. Grouping the tokens directly is
    equivalent for well-formed programs and correct for those.
    """
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps

# ---------------------------------------------------------------- execution --
def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    """Execute a tokenised program. Returns (invalid_flag, result)."""
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


# ------------------------------------------------------------------ program --
def equal_program(program1: List[str], program2: List[str]) -> bool:
    """Symbolic equivalence of gold (program1) and prediction (program2).

    Same protocol as the official implementation -- literals become symbols,
    table steps become opaque variables, and the two expressions are compared
    after `simplify`, so a differently-arranged but algebraically identical
    program still counts. A prediction may only use symbols that appear in gold,
    which is what stops it from introducing a constant of its own (the `100` of
    a percentage rescaling, say). Only the step-splitting differs: it groups
    tokens rather than splitting a joined string on ")".
    """
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


# ------------------------------------------------------------------ metrics --
def _coerce_answer(value):
    """ViNumQA stores exe_ans as a string; "yes"/"no" stay as they are."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    """(program_accuracy, execution_accuracy) for a single item.

    A generated_program that fails to tokenize (e.g. cut off mid-generation,
    missing a closing paren) scores (0.0, 0.0) rather than raising -- this is
    expected input from a real model, not a bug to surface as an exception.
    gold_program is assumed well-formed and is not caught the same way, so a
    malformed *gold* label still raises loudly instead of silently scoring 0.
    """
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


def evaluate_dataframe(df, generated_col: str = "generated_program",
                       gold_program_col: str = "program",
                       gold_answer_col: str = "answer",
                       table_col: str = "table_raw",
                       extract_first: bool = True):
    """Score a DataFrame, returning (df + per-row scores, summary).

    `table_col` must hold the raw table (list of rows); without it, programs
    naming a table row cannot execute and score 0 on EA.
    """
    df = df.copy()
    pa_scores, ea_scores = [], []

    for _, row in df.iterrows():
        table = row[table_col] if table_col in df.columns else None
        pa, ea = score_one(row[generated_col], row[gold_program_col],
                           row[gold_answer_col], table, extract_first)
        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    return df, {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }

In [ ]:
import gc
import glob, os
from pathlib import Path
from tqdm import tqdm

# No </think> handling anywhere in this cell -- the one deliberate departure from
# `sft-w-reasoning-trace-distill/gemma3-4b-eval-only.ipynb`, whose loop decodes to text and cuts at
# the last </think>. Gemma has no thinking mode and this adapter was trained on program-only
# labels, so everything past the prompt IS the program; the Qwen sibling's cut at token id 151668
# has nothing to cut here either.

# One prompt at a time, deliberately. Batching several prompts together requires left padding, and
# Unsloth's fast inference path derives token positions from the cache length rather than from the
# attention mask -- so padded rows decode at the wrong positions (measured elsewhere in this repo:
# that cost 0.6419 -> 0.6338 PA on a similar model). Extra votes come from num_return_sequences
# instead: those share a single prompt, so there is nothing to pad, and memory stays bounded by
# N_VOTES rather than by N_VOTES x batch.
#
# N_VOTES = 1 is the like-for-like number -- against the Qwen baseline and against the
# with-reasoning arm. Raising it multiplies runtime, but far less painfully than on that sibling:
# this adapter writes a bare program (tens of tokens), not a full reasoning trace, so k=5 over all
# 497 samples is a realistic single-session run here.
N_VOTES = 1

# 256, same cap as the Qwen baseline -- a program is a few dozen tokens and generation stops at
# <end_of_turn> anyway. Unlike the with-reasoning arm, a low cap here cannot silently hand a
# reasoning trace to the parser; a truncated program simply fails to tokenize and scores 0, which
# the "no parsable program" counter in the error breakdown below makes visible.
EVAL_MAX_NEW_TOKENS = 256

# Only used when N_VOTES > 1 -- Gemma-3's own recommended sampling settings, as elsewhere in this
# repo (Qwen's 0.7 / 0.8 / 20 are Qwen's; each model uses what its authors published).
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 64

CKPT_PATH = Path(f"/kaggle/working/eval_partial_{ADAPTER_VARIANT}_k{N_VOTES}.csv")
CHECKPOINT_EVERY = 20   # samples between checkpoint writes

if "generated_program" not in test_df.columns:
    test_df["generated_program"] = ""

# Resume: this session's own /kaggle/working first, then any attached dataset -- an interrupted run
# whose output was attached as a data source resumes just as well as one that never lost its kernel.
# The legacy pattern is kept so checkpoints written before this notebook adopted the
# variant+k naming still resume instead of silently regenerating all 497 samples.
_ckpt_src = None
if CKPT_PATH.exists():
    _ckpt_src = str(CKPT_PATH)
else:
    for _pattern in (
        f"/kaggle/input/**/eval_partial_{ADAPTER_VARIANT}_k{N_VOTES}*.csv",
        "/kaggle/input/**/gemma3-4b-sft-wo-reasoning-test-checkpoint.csv",   # legacy name
    ):
        _hits = glob.glob(_pattern, recursive=True)
        if _hits:
            _ckpt_src = _hits[0]
            break
if _ckpt_src:
    _done = pd.read_csv(_ckpt_src, index_col=0)["generated_program"].fillna("").astype(str)
    _done = _done[_done.index.isin(test_df.index)]
    test_df.loc[_done.index, "generated_program"] = _done
    print(f"Resumed {int((test_df['generated_program'] != '').sum())} / {len(test_df)} from {_ckpt_src}")


def build_prompt(row):
    """Row -> templated prompt text, byte-identical to what training saw.

    tokenize=False only. On a Gemma3Processor the tokenize=True path is multimodal-only: it assumes
    every message["content"] is a LIST of content parts and raises "TypeError: string indices must
    be integers" on a plain string. Tokenizing is done separately, in encode_prompt.
    """
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


def encode_prompt(text):
    """Prompt text -> model inputs, matching training's token sequence exactly.

    text_tokenizer, not `tokenizer`: the latter is a Gemma3Processor whose __call__ takes `images`
    first, so tokenizer([text]) binds the prompt to the wrong argument.

    add_special_tokens=False because the gemma-3 template already emits <bos>. The default (True)
    would prepend a second one, and every training sequence had exactly one -- the training notebook
    strips the template's <bos> and lets SFTTrainer re-add it for precisely this reason.

    Deliberately NOT truncated to MAX_SEQ_LENGTH: 2048 is a training-time budget, and the Qwen
    baseline feeds full-length prompts at eval too (Qwen3 has a 32k context, Gemma-3 a 128k one).
    Truncating here would make this arm answer from a clipped table the baseline saw whole.
    """
    return text_tokenizer(
        [text], return_tensors="pt", add_special_tokens=False,
    ).to(model.device)


def vote(candidates):
    """Most common program among candidates, keyed by normalised form so that
    cosmetically different but structurally identical programs share a vote."""
    cleaned = [extract_program(c) for c in candidates if c and c.strip()]
    if not cleaned:
        return ""
    keyed = {}
    for c in cleaned:
        try:
            key = str(program_tokenization(c))
        except Exception:
            key = c.strip()
        keyed.setdefault(key, []).append(c)
    best = max(keyed, key=lambda k: (len(keyed[k]), -cleaned.index(keyed[k][0])))
    return keyed[best][0]


todo = [i for i in test_df.index if not str(test_df.at[i, "generated_program"]).strip()]
print(f"{len(todo)} samples to generate.")

for n, df_index in enumerate(tqdm(todo, desc=f"Generating (k={N_VOTES})"), start=1):
    enc = encode_prompt(build_prompt(test_df.loc[df_index]))

    # do_sample=False is stated explicitly, not left to default. Gemma-3-it ships a
    # generation_config carrying top_k=64/top_p=0.95, and whatever do_sample it defaults to is the
    # model's choice, not ours -- leaving it implicit means the headline k=1 number might be a
    # sampled draw that does not reproduce between runs. Sampling is used only when voting needs it.
    #
    # eos_token_id is passed either way: Gemma-3 ends a turn with <end_of_turn>, not <eos>, and
    # generate() only stops on ids listed here. Without it every sample burns all 256 tokens.
    gen_kwargs = dict(max_new_tokens=EVAL_MAX_NEW_TOKENS, do_sample=False,
                      eos_token_id=_EOS_IDS)
    if N_VOTES > 1:
        gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          min_p=0, num_return_sequences=N_VOTES)

    try:
        with torch.no_grad():
            out = model.generate(**enc, **gen_kwargs)
        prompt_len = enc["input_ids"].shape[1]
        cands = [
            text_tokenizer.decode(r[prompt_len:].tolist(), skip_special_tokens=True).strip()
            for r in out
        ]
        test_df.at[df_index, "generated_program"] = vote(cands) if N_VOTES > 1 else cands[0]
        del enc, out
    except torch.cuda.OutOfMemoryError:
        # Caught rather than fatal: a run that is already hours in should not be lost to one long
        # prompt. A blank generation scores 0 on both metrics and is counted as "empty generation"
        # in the error breakdown below.
        print(f"  OOM on index {df_index}; leaving it blank (scored 0).")
        test_df.at[df_index, "generated_program"] = ""

    # Every 20 samples, not every sample: the CSV is rewritten whole each time, so at ~1s per sample
    # that write is pure overhead. Losing at most 20 generations to an interruption is fine.
    if n % CHECKPOINT_EVERY == 0 or n == len(todo):
        test_df[["generated_program"]].to_csv(CKPT_PATH)
        gc.collect()
        torch.cuda.empty_cache()

test_df[["generated_program"]].to_csv(CKPT_PATH)
print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} generated "
      f"(N_VOTES={N_VOTES}, {len(todo)} newly generated this run).")
print(f"Checkpoint: {CKPT_PATH}")

In [ ]:
import json

# table_col="table_raw" is not optional: without the raw table rows, every table_sum/table_average/
# table_max/table_min program fails to execute and scores 0 on EA even when it is exactly right.
df_scored, summary = evaluate_dataframe(test_df, table_col="table_raw")
print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}

print(f"\nGemma3-4B (SFT, program-only labels, k={N_VOTES}): "
      f"PA = {summary['program_accuracy']:.4f}  |  EA = {summary['execution_accuracy']:.4f}")
print(f"Qwen3-4B  (SFT, program-only labels):        PA = 0.6419  |  EA = 0.6439")

# Persist per-sample scores, not just the headline numbers: the breakdown below and any later
# comparison need them, and /kaggle/working is lost if the session ends without Save Version.
#
# Keyed by ADAPTER_VARIANT and k, matching the with-reasoning sibling's naming, so a k=1 and a
# later k=5 run sit side by side instead of overwriting each other.
RESULTS_PATH = f"/kaggle/working/eval_scored_{ADAPTER_VARIANT}_k{N_VOTES}.csv"
SUMMARY_PATH = f"/kaggle/working/eval_summary_{ADAPTER_VARIANT}_k{N_VOTES}.json"
df_scored[["question", "program", "answer", "generated_program", "pa_score", "ea_score"]].to_csv(RESULTS_PATH)
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "run": f"gemma3-4b-sft-{ADAPTER_VARIANT}",
        "model": "unsloth/gemma-3-4b-it",
        "method": "QLoRA SFT, program-only labels (no reasoning trace)",
        "adapter_dir": ADAPTER_DIR,
        "data_dir": str(DATA_DIR),
        "n_test": len(test_df),
        "max_seq_length": MAX_SEQ_LENGTH,
        # The decoding knobs, spelled out: a PA/EA number is only comparable against another arm if
        # the reader can see whether it was greedy or voted.
        "n_votes": N_VOTES,
        "decoding": "greedy (do_sample=False)" if N_VOTES == 1 else
                    f"self-consistency vote over {N_VOTES} samples",
        "max_new_tokens": EVAL_MAX_NEW_TOKENS,
        "sampling": None if N_VOTES == 1 else
                    {"temperature": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K},
        **summary,
    }, f, ensure_ascii=False, indent=2)
print("Per-sample scores:", RESULTS_PATH)
print("Summary:", SUMMARY_PATH)

### Where the errors are

PA and EA alone do not say whether a low score means "wrong arithmetic" or "did not produce a
program at all". This splits the failures so the write-up can say which.

In [ ]:
_gen = df_scored["generated_program"].fillna("").astype(str)
_extracted = _gen.map(extract_program)

n = len(df_scored)
n_empty = int((_gen.str.strip() == "").sum())
n_no_program = int((_extracted.str.strip() == "").sum()) - n_empty
n_pa = int(df_scored["pa_score"].sum())
n_ea = int(df_scored["ea_score"].sum())
n_ea_not_pa = int(((df_scored["ea_score"] == 1) & (df_scored["pa_score"] == 0)).sum())
n_pa_not_ea = int(((df_scored["pa_score"] == 1) & (df_scored["ea_score"] == 0)).sum())
_uses_table_op = df_scored["program"].astype(str).str.contains("table_")

print(f"samples                                  : {n}")
print(f"  empty generation                       : {n_empty}")
print(f"  non-empty but no parsable program      : {n_no_program}")
print(f"  program exactly right (PA)             : {n_pa}  ({n_pa / n:.4f})")
print(f"  answer right (EA)                      : {n_ea}  ({n_ea / n:.4f})")
print(f"  right answer via a different program   : {n_ea_not_pa}")
print(f"  right program that failed to execute   : {n_pa_not_ea}   <- if this is large, check table_raw")
print(f"\ngold programs using a table_* operator  : {int(_uses_table_op.sum())}")
print(f"  PA on those                            : {df_scored.loc[_uses_table_op, 'pa_score'].mean():.4f}")
print(f"  EA on those                            : {df_scored.loc[_uses_table_op, 'ea_score'].mean():.4f}")

print("\n--- 5 wrong samples ---")
for _, r in df_scored[df_scored["ea_score"] == 0].head(5).iterrows():
    print(f"Q    : {str(r['question'])[:110]}")
    print(f"gold : {r['program']}   (= {r['answer']})")
    print(f"gen  : {str(r['generated_program'])[:200]!r}\n")

### Self-consistency retest on the currently-wrong samples only

Same diagnostic as the with-reasoning arm's, so the two can be read against each other. It is not a
full self-consistency run: it takes the samples that scored `pa_score == 0 and ea_score == 0` above,
regenerates each at `k=5, do_sample=True`, votes on the normalised program, and re-scores just that
subset. The question it answers is whether the remaining errors are **random** (voting recovers a
meaningful fraction) or **systematic** (voting barely moves the needle) -- worth knowing before
committing to a full k=5 pass over all 497 or to reworking the training data.

Cheaper here than on the with-reasoning arm: this adapter emits a bare program rather than a full
reasoning trace, so 5 generations per wrong sample is minutes-to-an-hour on a T4, not most of a
session. Still, get the headline PA/EA above saved first -- the JSON checkpoint below makes this
resumable either way.

In [ ]:
import gc
import json
from pathlib import Path
from tqdm import tqdm

# Only the samples that are currently wrong on both metrics -- no point re-testing the ones that
# already scored.
wrong_idx = df_scored.index[(df_scored["pa_score"] == 0) & (df_scored["ea_score"] == 0)].tolist()
print(f"Re-testing {len(wrong_idx)} currently-wrong samples with self-consistency (k=5)...")

N_VOTES_RETEST = 5
TEMPERATURE_RETEST = 1.0   # Gemma-3's recommended sampling settings, as in the main loop
TOP_P_RETEST = 0.95
TOP_K_RETEST = 64

RETEST_CKPT = Path(f"/kaggle/working/retest_{ADAPTER_VARIANT}_k{N_VOTES_RETEST}_results.json")
retest_results = {}
if RETEST_CKPT.exists():
    retest_results = {int(k): v for k, v in json.loads(RETEST_CKPT.read_text(encoding="utf-8")).items()}
    print(f"Resumed {len(retest_results)} / {len(wrong_idx)} from checkpoint.")

retest_todo = [i for i in wrong_idx if i not in retest_results]

for n, df_index in enumerate(tqdm(retest_todo, desc=f"Self-consistency retest (k={N_VOTES_RETEST})")):
    enc = encode_prompt(build_prompt(test_df.loc[df_index]))   # same prompt path as the main loop

    try:
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=EVAL_MAX_NEW_TOKENS,
                do_sample=True, temperature=TEMPERATURE_RETEST,
                top_p=TOP_P_RETEST, top_k=TOP_K_RETEST, min_p=0,
                num_return_sequences=N_VOTES_RETEST,
                eos_token_id=_EOS_IDS,
            )
        prompt_len = enc["input_ids"].shape[1]
        cands = [
            text_tokenizer.decode(r[prompt_len:].tolist(), skip_special_tokens=True).strip()
            for r in out
        ]
        retest_results[df_index] = vote(cands)
        del enc, out
    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on index {df_index}; leaving it blank.")
        retest_results[df_index] = ""

    if n % 10 == 0:
        RETEST_CKPT.write_text(json.dumps(retest_results, ensure_ascii=False), encoding="utf-8")
        gc.collect()
        torch.cuda.empty_cache()

RETEST_CKPT.write_text(json.dumps(retest_results, ensure_ascii=False), encoding="utf-8")
print(f"Done. Retested {len(retest_results)} / {len(wrong_idx)}.")

In [ ]:
retest_df = test_df.loc[wrong_idx].copy()
retest_df["generated_program"] = [retest_results[i] for i in wrong_idx]

retest_scored, retest_summary = evaluate_dataframe(retest_df, table_col="table_raw")
n_fixed = int(((retest_scored["pa_score"] == 1) | (retest_scored["ea_score"] == 1)).sum())

print(f"Self-consistency (k={N_VOTES_RETEST}) on the {len(wrong_idx)} previously-wrong samples:")
print(f"  PA: {retest_summary['program_accuracy']:.4f}   EA: {retest_summary['execution_accuracy']:.4f}")
print()
print(f"  {n_fixed} / {len(wrong_idx)} flipped to correct (PA=1 or EA=1)")
print(f"  -> equivalent to +{100 * n_fixed / len(test_df):.2f} points on the full "
      f"{len(test_df)}-sample PA/EA if these replace the k=1 predictions")
print()
if n_fixed / len(wrong_idx) > 0.15:
    print("A meaningful fraction flipped -- the errors look at least partly random. "
          f"Worth running full self-consistency (k=5) over all {len(test_df)} samples next: set "
          "N_VOTES = 5 in the generation cell, which writes to its own k=5 checkpoint.")
else:
    print("Barely moved -- the errors look systematic, not sampling noise. "
          "Self-consistency alone is unlikely to help; look at the error breakdown above "
          "(table_* rows in particular) and at the training data instead.")

RETEST_RESULTS_PATH = f"/kaggle/working/retest_scored_{ADAPTER_VARIANT}_k{N_VOTES_RETEST}.csv"
retest_scored[["question", "program", "answer", "generated_program", "pa_score", "ea_score"]].to_csv(RETEST_RESULTS_PATH)
print("\nPer-sample retest scores:", RETEST_RESULTS_PATH)

## Done

`/kaggle/working` now holds (all keyed by `ADAPTER_VARIANT` and `k`, same naming as
`sft-w-reasoning-trace-distill/gemma3-4b-eval-only.ipynb`, so the two arms' outputs can sit in one
folder without colliding):

| artifact | purpose |
|---|---|
| `eval_summary_wo_reasoning_k1.json` | PA/EA plus the run configuration and decoding knobs |
| `eval_scored_wo_reasoning_k1.csv` | generated program + PA/EA per sample |
| `eval_partial_wo_reasoning_k1.csv` | raw generations, resumable |
| `retest_wo_reasoning_k5_results.json` | self-consistency retest generations, resumable |
| `retest_scored_wo_reasoning_k5.csv` | PA/EA per retested sample |

The last two only exist if the retest section was run.

**When reporting Gemma3-4B vs Qwen3-4B on this arm, state the knobs that are not model-intrinsic:**
the truncation rate at `MAX_SEQ_LENGTH = 2048` printed by the training notebook (both truncate at
2048, but the tokenizers differ so the rate does), and the fact that this run is 1 epoch with early
stopping against the baseline's fixed 3 epochs.

**One protocol difference from the Qwen baseline worth stating explicitly:** the headline number
here is greedy (`do_sample=False`), where that baseline decoded with its authors' recommended
sampling. Greedy is the reproducible choice and matches the with-reasoning arm, but it is not
identical decoding -- if the two must be compared at exactly equal settings, re-run this with
`N_VOTES = 1` swapped for sampled decoding, or re-run the baseline greedily.